# Node 2 — Format Validation playground

Demonstrates `FormatValidationNode` end-to-end:
rule-based ACCEPT / REJECT / MANUAL_REVIEW → gray-zone detection → SSE events → audit record.

Node 2 always runs **after** Node 1: it receives the `FileReceptionResult` produced by Node 1
and uses `detected_mime` to make its decision.

> **Kernel**: select `.venv` (Python 3.12) in the top-right kernel picker.

## 1 — Imports

In [ ]:
import asyncio

from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

from classiflow.database.base import Base
from classiflow.database.repositories.audit import SqlAuditRepository
from classiflow.events.broadcaster import EventBroadcaster
from classiflow.ingesta.domain.context import JobContext
from classiflow.ingesta.domain.results import FileReceptionResult
from classiflow.ingesta.mime import detect_mime
from classiflow.ingesta.nodes import FileReceptionNode
from classiflow.ingesta.nodes.node2_format_validation import FormatValidationNode
from classiflow.services.audit.service import AuditService

print("imports OK")

## 2 — Database setup

Creates the SQLite engine and ensures all tables exist.
The `session_factory` is reused by every section below.

In [ ]:
from pathlib import Path

import classiflow.settings as _settings_mod

# settings.py lives at src/classiflow/settings.py → parents[2] = project root
_project_root = Path(_settings_mod.__file__).parents[2]
_db_path = _project_root / "data" / "classiflow.db"
DB_URL = f"sqlite+aiosqlite:///{_db_path.as_posix()}"

print(f"DB path : {_db_path}")

engine = create_async_engine(DB_URL, echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)

async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

print("database ready")

## 3 — Node helpers

Node 2 needs a `FileReceptionResult` produced by Node 1.
`run_pipeline` chains them in order and returns both results.

In [ ]:
from sqlalchemy.ext.asyncio import AsyncSession


def make_node1(session: AsyncSession, broadcaster: EventBroadcaster) -> FileReceptionNode:
    return FileReceptionNode(
        audit=AuditService(SqlAuditRepository(session)),
        broadcaster=broadcaster,
        mime_detector=detect_mime,
    )


def make_node2(session: AsyncSession, broadcaster: EventBroadcaster) -> FormatValidationNode:
    return FormatValidationNode(
        audit=AuditService(SqlAuditRepository(session)),
        broadcaster=broadcaster,
    )


async def run_pipeline(
    job_id: str,
    filename: str,
    file_bytes: bytes | None,
    *,
    session: AsyncSession,
    broadcaster: EventBroadcaster,
) -> tuple[FileReceptionResult, object]:
    ctx = JobContext(job_id=job_id, filename=filename)
    reception = await make_node1(session, broadcaster).run(ctx, file_bytes)
    if not reception.passed:
        return reception, None
    validation = await make_node2(session, broadcaster).run(ctx, reception)
    return reception, validation


print("helpers ready")

## 4 — Run with a valid PDF → ACCEPT

A minimal syntactically valid PDF with matching MIME and `.pdf` extension.
Expected decision: **ACCEPT**, `passed=True`.

In [ ]:
MINIMAL_PDF = (
    b"%PDF-1.4\n1 0 obj\n<< /Type /Catalog >>\nendobj\n"
    b"xref\n0 1\n0000000000 65535 f\ntrailer\n<< /Size 1 >>\nstartxref\n9\n%%EOF"
)

async with session_factory() as session:
    reception, validation = await run_pipeline(
        "demo-a2-001",
        "sample.pdf",
        MINIMAL_PDF,
        session=session,
        broadcaster=EventBroadcaster(),
    )
    await session.commit()

print("=== Node 1 — File Reception ===")
print(f"  passed        : {reception.passed}")
print(f"  detected_mime : {reception.detected_mime}")
print()
print("=== Node 2 — Format Validation ===")
print(f"  passed        : {validation.passed}")
print(f"  decision      : {validation.decision}")
print(f"  used_slm      : {validation.used_slm}")
print(f"  rejection     : {validation.rejection_reason or '—'}")

## 5 — Inspect the audit records

Both nodes write an audit entry for the same `job_id`.

In [ ]:
async with session_factory() as session:
    records = await SqlAuditRepository(session).list_for_job("demo-a2-001")

print("=== Audit records ===")
for r in records:
    print(f"  node        : {r.node}")
    print(f"  event       : {r.event}")
    print(f"  duration_ms : {r.duration_ms} ms")
    print(f"  detail      : {r.detail}")
    print()

## 6 — Observe SSE events in real time

Both nodes emit `STARTED` then `PASSED`/`FAILED` through the shared `EventBroadcaster`.
Subscribe before calling the pipeline to capture all four events.

In [ ]:
broadcaster_sse = EventBroadcaster()
events: list[object] = []


async def collect() -> None:
    async for event in broadcaster_sse.subscribe("demo-a2-002"):
        events.append(event)
        print(f"  SSE → node={event.node}  status={event.status}")


async with session_factory() as session:
    collect_task = asyncio.create_task(collect())
    await asyncio.sleep(0)  # yield so collect() starts subscribing before run() emits

    await run_pipeline(
        "demo-a2-002",
        "sample.pdf",
        MINIMAL_PDF,
        session=session,
        broadcaster=broadcaster_sse,
    )
    await broadcaster_sse.close("demo-a2-002")
    await collect_task
    await session.commit()

print(f"\ncollected {len(events)} events (2 per node = 4 total)")

## 7 — Decision matrix

Each row exercises a different rule-based branch:

| Case | Expected decision |
|------|------------------|
| `.pdf` with PDF magic bytes | ACCEPT |
| `.html` file | REJECT (disabled extension) |
| Unknown MIME (ZIP header) | MANUAL_REVIEW |
| PDF bytes saved as `.docx` | gray zone → `NotImplementedError` (SLM pending T12) |

In [ ]:
HTML_BYTES = b"<html><body>Hola mundo</body></html>"
ZIP_BYTES = b"PK\x03\x04" + b"\x00" * 100  # ZIP magic bytes — unknown MIME for Node 2

cases = [
    ("pdf-accept", "sample.pdf", MINIMAL_PDF),
    ("html-reject", "page.html", HTML_BYTES),
    ("zip-unknown", "archive.zip", ZIP_BYTES),
    ("pdf-as-docx", "report.docx", MINIMAL_PDF),  # gray zone → NotImplementedError
]


async def _run_case(job_id: str, filename: str, data: bytes) -> tuple[object, object, str | None]:
    try:
        async with session_factory() as session:
            r, v = await run_pipeline(
                job_id,
                filename,
                data,
                session=session,
                broadcaster=EventBroadcaster(),
            )
            await session.commit()
    except NotImplementedError:
        return None, None, "SLM escalation pending (T12)"
    else:
        return r, v, None


print(f"{'job':20} {'filename':15} {'passed':7} {'decision':14} {'rejection'}")
print("-" * 75)

for job_id, filename, data in cases:
    reception, validation, error = await _run_case(job_id, filename, data)
    if error:
        print(f"{job_id:20} {filename:15} {'—':7} {'gray zone':14} {error}")
    elif validation is None:
        print(f"{job_id:20} {filename:15} {'False':7} {'—':14} node1 rejected")
    else:
        print(
            f"{job_id:20} {filename:15} {validation.passed!s:7}"
            f" {validation.decision.value:14} {validation.rejection_reason or '—'}"
        )

## 8 — Run with a real file from disk

Drop any PDF, DOCX, HTML, or image into:
```
src/classiflow/playground/samples/
```
then run the cell — it processes every file in that folder through the full Node 1 → Node 2 pipeline.

In [ ]:
from pathlib import Path

import IPython.display as ipyd
from IPython.display import HTML

samples_dir = next(
    p / "samples"
    for p in [Path.cwd(), Path.cwd() / "src" / "classiflow" / "playground"]
    if (p / "samples").is_dir()
)
files = sorted(samples_dir.iterdir())
if not files:
    msg = f"No files in {samples_dir}. Drop a PDF/DOCX/image there first."
    raise FileNotFoundError(msg)


async def _run_real_file(
    job_id: str, file_path: Path, file_bytes: bytes
) -> tuple[object, object, object, str | None]:
    try:
        async with session_factory() as session:
            repo = SqlAuditRepository(session)
            r, v = await run_pipeline(
                job_id,
                file_path.name,
                file_bytes,
                session=session,
                broadcaster=EventBroadcaster(),
            )
            a2_rec = next(
                (
                    rec
                    for rec in await repo.list_for_job(job_id)
                    if rec.node == "node2_format_validation"
                ),
                None,
            )
            await session.commit()
    except NotImplementedError:
        return None, None, None, "gray zone — SLM escalation pending (T12)"
    else:
        return r, v, a2_rec, None


for file_path in files:
    file_bytes = file_path.read_bytes()
    job_id = f"demo-real-{file_path.stem}"

    reception, validation, a2_record, error_msg = await _run_real_file(
        job_id, file_path, file_bytes
    )

    if error_msg:
        decision_str = "gray zone"
        status_icon = "⚠️ GRAY ZONE"
        status_color = "#e65100"
        rejection_str = error_msg
        duration_str = "—"
        used_slm_str = "—"
    elif validation is None:
        decision_str = "—"
        status_icon = "❌ FAILED (node1)"
        status_color = "#c62828"
        rejection_str = reception.rejection_reason
        duration_str = "—"
        used_slm_str = "—"
    else:
        decision_str = validation.decision.value
        status_icon = "✅ PASSED" if validation.passed else "❌ FAILED"
        status_color = "#2e7d32" if validation.passed else "#c62828"
        rejection_str = validation.rejection_reason or "—"
        duration_str = f"{a2_record.duration_ms} ms" if a2_record else "—"
        used_slm_str = str(validation.used_slm)

    rows = [
        ("File", file_path.name),
        ("Size", f"{len(file_bytes) / 1024:.1f} KB"),
        ("Decision", decision_str),
        ("Used SLM", used_slm_str),
        ("Duration (node2)", duration_str),
        ("Rejection reason", rejection_str),
    ]

    rows_html = "".join(
        f'<tr><td style="color:#555;padding:4px 12px 4px 0;white-space:nowrap">'
        f"{k}</td>"
        f'<td style="font-family:monospace;padding:4px 0">{v}</td></tr>'
        for k, v in rows
    )

    ipyd.display(
        HTML(f"""
        <div style="border:1px solid #ddd;border-radius:8px;padding:16px;
                    margin:8px 0;font-family:sans-serif;max-width:540px">
          <div style="font-size:1.1em;font-weight:bold;color:{status_color};
                      margin-bottom:10px">{status_icon}</div>
          <table style="border-collapse:collapse;width:100%">{rows_html}</table>
        </div>
        """)
    )

## 9 — Cleanup

In [ ]:
await engine.dispose()
print("engine disposed — database file is now unlocked")